pip install gym-simplegrid
pip install minigrid
pip install accelerate

In [1]:
# IMPORTS

from __future__ import annotations
from minigrid.core.constants import COLOR_NAMES
from minigrid.core.grid import Grid
from minigrid.core.mission import MissionSpace
from minigrid.core.world_object import Door, Goal, Key, Wall
from minigrid.minigrid_env import MiniGridEnv
from minigrid.envs import EmptyEnv
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

C:\Users\Fcomm\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
#CREATE ENVIRONMENT

class SimpleEnv(MiniGridEnv):
    def __init__(self, map_type=0, size=7, agent_start_pos=(1,1), agent_start_dir=0, max_steps=None, **kwargs):
        self.map_type = map_type
        self.agent_start_pos = agent_start_pos
        self.agent_start_dir = agent_start_dir

        mission_space = MissionSpace(mission_func=self._gen_mission)

        if max_steps is None:
            max_steps = 4 * size**2

        super().__init__(
            mission_space=mission_space,
            grid_size=size,
            see_through_walls=True,
            max_steps=max_steps,
            **kwargs,
    )
    
    @staticmethod
    def _gen_mission():
        return "grand mission"

    def _gen_grid(self, width, height):

        self.grid = Grid(width, height)
        self.grid.wall_rect(0, 0, width, height)

        if self.map_type == 0:
            # simple open map
            pass

        elif self.map_type == 1:
            mid_y = height // 2 
            gap_x = width // 2    
            for x in range(width):
                if x == gap_x:
                    continue 
                self.grid.set(x, mid_y, Wall())

        elif self.map_type == 2:
            # small maze block
            self.grid.wall_rect(2, 2, 3, 3)

        elif self.map_type == 3:
            # dead end test
            self.grid.set(2, 1, Wall())
            self.grid.set(2, 2, Wall())
            self.grid.set(2, 3, Wall())

        # goal always placed
        self.put_obj(Goal(), width - 2, height - 2)

        self.agent_pos = self.agent_start_pos
        self.agent_dir = self.agent_start_dir
        
        self.mission = "grand mission"

In [3]:
#LLM caller
class LLM:

    def __init__(self):

        model_name = "google/flan-t5-small"

        print("Loading tokenizer...")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)

        print("Loading model...")
        self.model = AutoModelForSeq2SeqLM.from_pretrained(model_name)


    def ask(self, prompt):

        inputs = self.tokenizer(
            prompt,
            return_tensors="pt",
            truncation=True,
            max_length=256
        )


        outputs = self.model.generate(
            **inputs,
            max_new_tokens=20,
            do_sample=True,
            temperature=0.5,
            top_p=0.9,
            repetition_penalty=1.2,
            no_repeat_ngram_size=2
        )
        response = self.tokenizer.decode(outputs[0], skip_special_tokens=True)

        return response

In [4]:
#Relay
class Relay:

    def __init__(self, llm):

        self.llm = llm

        self.OBJECTS = {
            0: "unseen",
            1: "empty",
            2: "wall",
            3: "floor",
            4: "door",
            5: "key",
            6: "ball",
            7: "box",
            8: "goal"
        }


    def get_local_vision(self, obs):

        grid = obs["image"]

        vision = {
            "Front": grid[3][4],
            "Left": grid[2][3],
            "Right": grid[4][3],
        }

        return vision


    def decode_tile(self, tile):
        obj = self.OBJECTS.get(tile[0], "unknown")
        return obj


    def build_prompt(self, vision):
        description = ""

        for pos, tile in vision.items():
            description += f"{pos}: {self.decode_tile(tile)}\n"

        prompt = f"""                
**Rules**
You are a navigation agent operating in a 2D grid-based world.
Your ultimate mission is to reach the "goal tile".
You MUST do so via the most optimal sequence of moves.

Your perceptive field is limited to 3 tiles around you:
- Ahead: the tile directly in front of you
- Left: the tile directly to your left
- Right: the tile directly to your right

**Movement rules:**
- You CANNOT enter wall tiles
- You CAN move onto floor and goal tiles

**Reasoning Protocol — follow each step in order:**
1. Build, in your working memory, a map of the place as you explore it.
2. Assess immediate moves: Which of the valid actions are physically possible right now?
3. Evaluate goal proximity: Which passable move most plausibly advances toward an unknown goal, given no walls are blocking that corridor?
4. Select optimal action: Choose the single action with the best forward progress potential while avoiding immediate dead ends.
5. If all forward paths are blocked, turn to explore a different direction.
6. If you see a goal, move toward it.


Environment:
{description}
**Output format:**
Only reply with either of these three choices:
- [forward]
- [left]
- [right]   
"""
                
        #print  (prompt)
        return prompt


    def decide_action(self, obs):

        vision = self.get_local_vision(obs)

        prompt = self.build_prompt(vision)

        response = self.llm.ask(prompt)

        print("LLM response:", response)

        return self.text_to_action(response)


    def text_to_action(self, response):

        response = response.lower().strip()

        if "left" in response:
            return 0
        elif "right" in response:
            return 1
        elif "forward" in response or "ahead" in response:
            return 2
        else:
            print("Unknown response, no movement")
            return 3

In [5]:
def main():
    llm = LLM()
    relay = Relay(llm)

    for map_type in range(4):
        print(f"\n=== Testing map {map_type} ===")

        env = SimpleEnv(map_type=map_type, render_mode="human")
        obs, info = env.reset()

        done = False
        limitor = 0

        while not done:
            action = relay.decide_action(obs)
            if action == 3:
                limitor += 1
                continue

            obs, reward, terminated, truncated, info = env.step(action)
            done = terminated or truncated
            print("=================================================")

            limitor += 1
            if limitor == 10:
                break

        env.close()

In [6]:
main()

Loading tokenizer...
Loading model...


Loading weights: 100%|██████████| 190/190 [00:00<00:00, 4516.52it/s]
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning



=== Testing map 0 ===
LLM response: Ahead: the tile directly to your left
LLM response: Ahead
LLM response: Ahead
LLM response: Ahead
LLM response: Ahead: the tile directly to your left
LLM response: Move onto floor and goal tiles.
Unknown response, no movement
LLM response: You CAN move onto floor and goal tiles
Unknown response, no movement
LLM response: Ahead
LLM response: You can move onto floor and goal tiles.
Unknown response, no movement
LLM response: The map is designed to be a map of the place.
Unknown response, no movement
LLM response: The tile directly to your left is the tile direct to the right.
LLM response: Move onto floor and goal tiles.
Unknown response, no movement
LLM response: Ahead
LLM response: Ahead: the tile directly in front of you.
LLM response: Ahead: the tile directly to your left - Aside: a tile direct to you
LLM response: Ahead: the tile directly in front of you
LLM response: Move onto the goal tile.
Unknown response, no movement
LLM response: Ahead
LLM 